# Bear + Discord on Google Colab
This notebook builds the headless Rust runner from this fork and starts the Discord control bot. Select a GPU runtime first. Add `DISCORD_BOT_TOKEN` in Colab Secrets.

In [ ]:
# System packages and Rust toolchain
!apt-get update -qq
!apt-get install -y -qq build-essential curl wget file libssl-dev libgtk-3-dev libayatana-appindicator3-dev librsvg2-dev patchelf libwebkit2gtk-4.1-dev
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] = '/root/.cargo/bin:' + os.environ['PATH']


In [ ]:
# Clone the user's fork and the integration branch
%cd /content
!git clone --branch feature/colab-discord-headless https://github.com/Nader-Geek/Bear-BOT.git
%cd /content/Bear-BOT


In [ ]:
# Download Bear's pinned local ONNX models and runtime (several hundred MB)
!bash scripts/fetch-models.sh
!bash scripts/fetch-runtime.sh
import glob
ort_candidates = glob.glob('/content/Bear-BOT/runtimes/**/libonnxruntime.so*', recursive=True)
assert ort_candidates, 'ONNX Runtime library was not found after fetch-runtime.sh'
os.environ['BEAR_MODELS_DIR'] = '/content/Bear-BOT/models'
os.environ['BEAR_ORT_PATH'] = ort_candidates[0]
print('ONNX Runtime:', os.environ['BEAR_ORT_PATH'])


In [ ]:
# Build the headless runner. This uses Bear's existing Pipeline, not a Python rewrite.
!cargo build --release -p bear-runner


In [ ]:
# Install the Discord control plane
!pip -q install -r bot/requirements.txt
from google.colab import userdata
token = userdata.get('DISCORD_BOT_TOKEN')
assert token, 'Add DISCORD_BOT_TOKEN in Colab > Secrets and grant this notebook access.'
os.environ['DISCORD_BOT_TOKEN'] = token
os.environ['BEAR_RUNNER'] = '/content/Bear-BOT/target/release/bear-runner'


## Optional: local Qwen-Image-Edit-2511
Bear's existing sidecar supports Qwen-Image-Edit-2511 through SDNQ, but its quantized weights are roughly 18.16 GB before runtime memory. **The current headless runner does not yet pass the desktop app's validated sidecar settings**, so installing the weights here will not make `!clean` use Qwen. Do not download them until headless sidecar activation is implemented and the available Colab GPU has been checked.

In [ ]:
# Start the bot. Keep this cell running while you use Discord.
!python bot/bear_bot.py
